# 04 Double Sérsic PSF-Convolved Mock

Validate recovery of bulge+disk parameters using CompositeModel, ConvolvedModel, and Dynesty.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from baymorph import (
    CompositeModel,
    ConvolvedModel,
    DynestyFitter,
    GaussianPSF,
    Parameter,
    ParameterSet,
    SersicModel,
    add_gaussian_noise,
    make_constant_noise_map,
    plot_data_model_residual,
)

In [ ]:
from matplotlib.colors import LogNorm

BAYMORPH_IMAGE_CMAP = "turbo"


def _log_limits(
    *images,
    lower_percentile=0.5,
    upper_percentile=99.9,
    floor=1e-3,
):
    """Return robust positive limits for logarithmic image display."""
    positive_values = []

    for image in images:
        values = np.asarray(
            image,
            dtype=float,
        )
        values = values[
            np.isfinite(values)
            & (values > 0)
        ]

        if values.size > 0:
            positive_values.append(values)

    if len(positive_values) == 0:
        return floor, 1.0

    positive_values = np.concatenate(
        positive_values,
    )

    vmin = np.percentile(
        positive_values,
        lower_percentile,
    )
    vmax = np.percentile(
        positive_values,
        upper_percentile,
    )

    vmin = max(
        float(vmin),
        floor,
    )
    vmax = max(
        float(vmax),
        vmin * 10,
    )

    return vmin, vmax


def show_log_image(
    image,
    title,
    cmap=BAYMORPH_IMAGE_CMAP,
    colorbar_label="Flux",
):
    """Display an image using a robust logarithmic normalization."""
    vmin, vmax = _log_limits(
        image,
    )

    display_image = np.clip(
        np.asarray(
            image,
            dtype=float,
        ),
        vmin,
        vmax,
    )

    plt.figure(
        figsize=(6, 6),
    )

    plt.imshow(
        display_image,
        origin="lower",
        cmap=cmap,
        norm=LogNorm(
            vmin=vmin,
            vmax=vmax,
        ),
    )

    plt.colorbar(
        label=colorbar_label,
    )

    plt.title(
        f"{title} (log scale)",
    )

    plt.show()


def plot_log_data_model_residual(
    data,
    model,
    title="",
    cmap=BAYMORPH_IMAGE_CMAP,
):
    """Plot data and model on a log scale, with a linear residual panel."""
    data = np.asarray(
        data,
        dtype=float,
    )

    model = np.asarray(
        model,
        dtype=float,
    )

    residual = data - model

    vmin, vmax = _log_limits(
        data,
        model,
    )

    data_display = np.clip(
        data,
        vmin,
        vmax,
    )

    model_display = np.clip(
        model,
        vmin,
        vmax,
    )

    residual_limit = np.nanpercentile(
        np.abs(residual),
        99,
    )

    if not np.isfinite(residual_limit) or residual_limit == 0:
        residual_limit = 1.0

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(15, 4),
    )

    data_im = axes[0].imshow(
        data_display,
        origin="lower",
        cmap=cmap,
        norm=LogNorm(
            vmin=vmin,
            vmax=vmax,
        ),
    )
    axes[0].set_title(
        "Data (log)"
    )
    fig.colorbar(
        data_im,
        ax=axes[0],
    )

    model_im = axes[1].imshow(
        model_display,
        origin="lower",
        cmap=cmap,
        norm=LogNorm(
            vmin=vmin,
            vmax=vmax,
        ),
    )
    axes[1].set_title(
        "Model (log)"
    )
    fig.colorbar(
        model_im,
        ax=axes[1],
    )

    residual_im = axes[2].imshow(
        residual,
        origin="lower",
        cmap="RdBu_r",
        vmin=-residual_limit,
        vmax=residual_limit,
    )
    axes[2].set_title(
        "Residual"
    )
    fig.colorbar(
        residual_im,
        ax=axes[2],
    )

    if title:
        fig.suptitle(
            title,
            y=1.02,
        )

    plt.tight_layout()
    plt.show()


In [ ]:
true_flux_bulge = 3000
true_re_bulge = 3

bulge = SersicModel(
    flux=true_flux_bulge,
    re=true_re_bulge,
    n=4,
    q=0.8,
    theta=0.0,
    x0=30,
    y0=30,
)

In [ ]:
true_flux_disk = 7000
true_re_disk = 12

disk = SersicModel(
    flux=true_flux_disk,
    re=true_re_disk,
    n=1,
    q=0.8,
    theta=0.0,
    x0=30,
    y0=30,
)

In [ ]:
true_model = CompositeModel([bulge, disk])

psf = GaussianPSF(fwhm=3.0)

true_model_psf = ConvolvedModel(
    true_model,
    psf,
)

In [ ]:
clean_image = true_model_psf.render(shape=(60, 60))

data = add_gaussian_noise(
    clean_image,
    sigma=1.0,
    seed=42,
)

noise = make_constant_noise_map(
    data.shape,
    sigma=1.0,
)

In [ ]:
show_log_image(
    data,
    title="PSF-convolved double-Sérsic mock",
)

In [ ]:
print("Bulge flux:", true_flux_bulge)
print("Bulge re:", true_re_bulge)

print()

print("Disk flux:", true_flux_disk)
print("Disk re:", true_re_disk)

In [ ]:
def model_factory(
    flux_bulge,
    re_bulge,
    flux_disk,
    re_disk,
):
    bulge = SersicModel(
        flux=flux_bulge,
        re=re_bulge,
        n=4,
        q=0.8,
        theta=0.0,
        x0=30,
        y0=30,
    )

    disk = SersicModel(
        flux=flux_disk,
        re=re_disk,
        n=1,
        q=0.8,
        theta=0.0,
        x0=30,
        y0=30,
    )

    composite = CompositeModel([bulge, disk])

    return ConvolvedModel(
        composite,
        psf,
    )

In [ ]:
parameters = ParameterSet(
    [
        Parameter("flux_bulge", 500, 6000),
        Parameter("re_bulge", 1, 8),
        Parameter("flux_disk", 3000, 12000),
        Parameter("re_disk", 5, 20),
    ]
)

In [ ]:
fitter = DynestyFitter(
    model_factory=model_factory,
    data=data,
    noise=noise,
    parameters=parameters,
)

In [ ]:
result = fitter.run(
    nlive=150,
    sample="rwalk",
    print_progress=True,
)

In [ ]:
result.best_parameters

In [ ]:
for parameter in [
    "flux_bulge",
    "re_bulge",
    "flux_disk",
    "re_disk",
]:
    print(parameter)
    print(result.summary(parameter))
    print()

In [ ]:
best = result.best_parameters

best_model = model_factory(
    flux_bulge=best["flux_bulge"],
    re_bulge=best["re_bulge"],
    flux_disk=best["flux_disk"],
    re_disk=best["re_disk"],
)

best_image = best_model.render(
    shape=data.shape,
)

In [ ]:
plot_log_data_model_residual(
    data,
    best_image,
    title="PSF-convolved double-Sérsic 4D best fit",
)

In [ ]:
print("Bulge flux")
print(true_flux_bulge, best["flux_bulge"])

print()
print("Bulge re")
print(true_re_bulge, best["re_bulge"])

print()
print("Disk flux")
print(true_flux_disk, best["flux_disk"])

print()
print("Disk re")
print(true_re_disk, best["re_disk"])

In [ ]:
print(
    "CompositeModel + ConvolvedModel successfully recover "
    "flux and effective radius for both Sérsic components."
)